## Imports

In [ ]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, ConcatDataset
from torchvision import transforms
from torchvision.utils import save_image
import medmnist
from medmnist import INFO
import numpy as np
from pytorch_fid import fid_score
from tqdm import tqdm
import torch.nn.utils as utils
import shutil
import itertools
import json
import copy
import matplotlib.pyplot as plt  # Adicionado para plotagem
import seaborn as sns  # Adicionado para gráficos mais estéticos

class Config:
    def __init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.data_flag = 'bloodmnist'
        self.batch_size = 128
        self.latent_dim = 100
        self.num_epochs = 200
        self.n_samples_save = 10000
        self.seeds = [42, 123, 456, 789, 999]
        self.patience = 10
        self.disc_train_ratio = 2
        self.fid_eval_interval = 50  # Avaliar FID a cada 50 épocas
        self.lr_G = 0.0001
        self.lr_D = 0.0004
        self.beta1 = (0.5, 0.999)
        self.output_dir = "./experiments"
        
        info = INFO[self.data_flag]
        self.task = info['task']
        self.n_channels = info['n_channels']
        self.n_classes = len(info['label'])
        self.DataClass = getattr(medmnist, info['python_class'])
    
    def to_dict(self):
        def serialize(val):
            if isinstance(val, torch.device):
                return str(val)
            elif isinstance(val, tuple):
                return list(val)
            elif isinstance(val, type):
                return val.__name__
            return val

        return {k: serialize(v) for k, v in self.__dict__.items() if not k.startswith('__') and not callable(v)}
    
    def update_from_dict(self, params):
        for key, value in params.items():
            if hasattr(self, key):
                setattr(self, key, value)

## Data

In [ ]:
class DataHandler:
    def __init__(self, config):
        self.config = config
        self.transform = transforms.Compose([
            transforms.Resize(32),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])
    
    def load_datasets(self):
        train = self.config.DataClass(split='train', transform=self.transform, download=True)
        val = self.config.DataClass(split='val', transform=self.transform, download=True)
        test = self.config.DataClass(split='test', transform=self.transform, download=True)
        return ConcatDataset([train, val, test])
    
    def get_dataloader(self, dataset):
        return DataLoader(
            dataset, 
            batch_size=self.config.batch_size, 
            shuffle=True,
            num_workers=4,
            pin_memory=True
        )

## Conditional Generator

In [ ]:
class Generator(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.label_emb = nn.Embedding(config.n_classes, 100)
        self.init_size = 8
        
        self.l1 = nn.Sequential(
            nn.Linear(config.latent_dim + 100, 512 * self.init_size ** 2),
            nn.BatchNorm1d(512 * self.init_size ** 2),
            nn.ReLU(inplace=True)
        )
        
        self.conv_blocks = nn.Sequential(
            nn.ConvTranspose2d(512, 256, 4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, config.n_channels, 3, stride=1, padding=1),
            nn.Tanh()
        )

    def forward(self, noise, labels):
        label_embedding = self.label_emb(labels)
        x = torch.cat((noise, label_embedding), dim=1)
        x = self.l1(x)
        x = x.view(-1, 512, self.init_size, self.init_size)
        return self.conv_blocks(x)

## Conditional Discriminator

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.label_emb = nn.Embedding(config.n_classes, 32*32)
        
        self.conv1 = utils.spectral_norm(nn.Conv2d(config.n_channels + 1, 64, 4, 2, 1))
        self.leakyrelu1 = nn.LeakyReLU(0.2, inplace=True)
        
        self.conv2 = utils.spectral_norm(nn.Conv2d(64, 128, 4, 2, 1))
        self.leakyrelu2 = nn.LeakyReLU(0.2, inplace=True)
        
        self.conv3 = utils.spectral_norm(nn.Conv2d(128, 256, 4, 2, 1))
        self.leakyrelu3 = nn.LeakyReLU(0.2, inplace=True)
        
        self.flatten = nn.Flatten()
        self.fc = utils.spectral_norm(nn.Linear(256*4*4, 1))

    def forward(self, img, labels):
        label_emb = self.label_emb(labels).view(-1, 1, 32, 32)
        x = torch.cat((img, label_emb), dim=1)
        x = self.leakyrelu1(self.conv1(x))
        x = self.leakyrelu2(self.conv2(x))
        x = self.leakyrelu3(self.conv3(x))
        x = self.flatten(x)
        return self.fc(x)

## Treino

In [ ]:
class GANTrainer:
    def __init__(self, config):
        self.config = config
        self.data_handler = DataHandler(config)
        self._initialize_components()
        
    def _initialize_components(self):
        self.dataset = self.data_handler.load_datasets()
        self.dataloader = self.data_handler.get_dataloader(self.dataset)
        
        self.generator = Generator(self.config).to(self.config.device)
        self.discriminator = Discriminator(self.config).to(self.config.device)
        
        self.optimizer_G = optim.Adam(
            self.generator.parameters(), 
            lr=self.config.lr_G, 
            betas=self.config.beta1
        )
        self.optimizer_D = optim.Adam(
            self.discriminator.parameters(), 
            lr=self.config.lr_D, 
            betas=self.config.beta1
        )
        self.criterion = nn.BCEWithLogitsLoss()

    def train_epoch(self):
        d_losses, g_losses = [], []
        self.generator.train()
        self.discriminator.train()

        for i, (imgs, labels) in enumerate(self.dataloader):
            imgs = imgs.to(self.config.device)
            labels = labels.squeeze().long().to(self.config.device)
            batch_size = imgs.size(0)
            
            # Treinar Discriminador
            self.optimizer_D.zero_grad()

            valid = torch.rand(batch_size, 1, device=self.config.device) * 0.2 + 0.9
            fake = torch.full((batch_size, 1), 0.1, device=self.config.device)

            real_pred = self.discriminator(imgs, labels)
            d_real_loss = self.criterion(real_pred, valid)
            
            noise = torch.randn(batch_size, self.config.latent_dim, device=self.config.device)
            gen_imgs = self.generator(noise, labels)
            
            fake_pred = self.discriminator(gen_imgs.detach(), labels)
            d_fake_loss = self.criterion(fake_pred, fake)
            
            d_loss = (d_real_loss + d_fake_loss) / 2
            d_loss.backward()
            utils.clip_grad_norm_(self.discriminator.parameters(), 1.0)
            self.optimizer_D.step()

            # Treinar Gerador
            if i % self.config.disc_train_ratio == 0:
                self.optimizer_G.zero_grad()
                gen_imgs = self.generator(noise, labels)
                g_pred = self.discriminator(gen_imgs, labels)
                g_loss = self.criterion(g_pred, valid)
                g_loss.backward()
                utils.clip_grad_norm_(self.generator.parameters(), 1.0)
                self.optimizer_G.step()
            
            d_losses.append(d_loss.item())
            g_losses.append(g_loss.item() if i % self.config.disc_train_ratio == 0 else g_losses[-1])
        
        return np.mean(d_losses), np.mean(g_losses)

    def generate_samples(self, output_dir, n_samples=10000):
        self.generator.eval()
        if os.path.exists(output_dir):
            shutil.rmtree(output_dir)
        os.makedirs(output_dir, exist_ok=True)
        
        with torch.no_grad():
            generated_count = 0
            samples_per_class = max(1, n_samples // self.config.n_classes)

            for class_idx in tqdm(range(self.config.n_classes), desc="Generating samples per class"):
                current_class_samples_generated = 0
                while current_class_samples_generated < samples_per_class:
                    batch_size_to_generate = min(self.config.batch_size, samples_per_class - current_class_samples_generated)
                    if batch_size_to_generate == 0:
                        break

                    noise = torch.randn(batch_size_to_generate, self.config.latent_dim, device=self.config.device)
                    labels = torch.full((batch_size_to_generate,), class_idx, dtype=torch.long, device=self.config.device)
                    
                    gen_imgs = (self.generator(noise, labels) + 1) / 2
                    
                    for i in range(batch_size_to_generate):
                        save_image(gen_imgs[i], os.path.join(output_dir, f"{generated_count}.png"))
                        generated_count += 1
                    current_class_samples_generated += batch_size_to_generate
        self.generator.train()

class ExperimentRunner:
    def __init__(self, config):
        self.config = config
        self.real_samples_dir = os.path.join(config.output_dir, "real_samples")
        self.generated_dir = os.path.join(config.output_dir, "generated_samples")
        
    def _save_real_samples(self):
        if os.path.exists(self.real_samples_dir) and len(os.listdir(self.real_samples_dir)) >= self.config.n_samples_save:
            print(f"Found existing real samples in {self.real_samples_dir}. Skipping regeneration.")
            return
            
        print("Saving real samples for FID calculation...")
        if os.path.exists(self.real_samples_dir):
            shutil.rmtree(self.real_samples_dir)
        os.makedirs(self.real_samples_dir, exist_ok=True)
        
        dataset = DataHandler(self.config).load_datasets()
        num_real_samples_to_save = min(self.config.n_samples_save, len(dataset))
        
        for i in tqdm(range(num_real_samples_to_save), desc="Saving real samples"):
            img, _ = dataset[i]
            save_image((img + 1) / 2, os.path.join(self.real_samples_dir, f"{i}.png"))
        print(f"Finished saving {num_real_samples_to_save} real samples.")

    def _calculate_fid(self, generated_dir):
        try:
            fid = fid_score.calculate_fid_given_paths(
                [self.real_samples_dir, generated_dir],
                batch_size=64,
                device=self.config.device,
                dims=2048
            )
            return fid
        except Exception as e:
            print(f"Error during FID calculation: {e}")
            return float('inf')
    
    def _plot_training_curves(self, results, exp_dir):
        """Gera gráficos de perdas e FID durante o treinamento"""
        epochs = [e['epoch'] for e in results['epoch_details']]
        d_losses = [e['d_loss'] for e in results['epoch_details']]
        g_losses = [e['g_loss'] for e in results['epoch_details']]
        fid_epochs = [e['epoch'] for e in results['fid_details']]
        fid_values = [e['fid'] for e in results['fid_details']]
        
        # Configurações estéticas
        sns.set_style("whitegrid")
        plt.figure(figsize=(15, 10))
        
        # Gráfico de Perdas
        plt.subplot(2, 1, 1)
        plt.plot(epochs, d_losses, 'b-', linewidth=1.5, label='Discriminador')
        plt.plot(epochs, g_losses, 'r-', linewidth=1.5, label='Gerador')
        plt.title('Evolução das Perdas durante o Treinamento', fontsize=14)
        plt.xlabel('Época', fontsize=12)
        plt.ylabel('Loss', fontsize=12)
        plt.legend(fontsize=12)
        plt.grid(True, alpha=0.3)
        plt.ylim(0, max(max(d_losses), max(g_losses)) * 1.1)
        
        # Gráfico de FID
        plt.subplot(2, 1, 2)
        plt.plot(fid_epochs, fid_values, 'g-o', linewidth=2, markersize=8)
        plt.title('Evolução do FID durante o Treinamento', fontsize=14)
        plt.xlabel('Época', fontsize=12)
        plt.ylabel('FID', fontsize=12)
        plt.grid(True, alpha=0.3)
        plt.ylim(0, max(fid_values) * 1.1)
        
        # Melhor FID
        best_fid_idx = np.argmin(fid_values)
        best_fid = fid_values[best_fid_idx]
        best_epoch = fid_epochs[best_fid_idx]
        plt.annotate(f'Melhor FID: {best_fid:.2f} @ época {best_epoch}', 
                    xy=(best_epoch, best_fid),
                    xytext=(best_epoch, best_fid + max(fid_values)*0.1),
                    arrowprops=dict(facecolor='black', shrink=0.05),
                    fontsize=12)
        
        plt.tight_layout()
        plt.savefig(os.path.join(exp_dir, 'training_metrics.png'))
        plt.close()

    def run_experiment(self, seed, exp_name):
        torch.manual_seed(seed)
        np.random.seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False

        exp_dir = os.path.join(self.config.output_dir, exp_name)
        os.makedirs(exp_dir, exist_ok=True)

        with open(os.path.join(exp_dir, "config.json"), "w") as f:
            json.dump(self.config.to_dict(), f, indent=2)

        self.generated_dir = os.path.join(exp_dir, "generated_samples")
        
        trainer = GANTrainer(self.config)
        results = {
            "training_time": 0,
            "final_fid": float('inf'),
            "epoch_details": [],
            "fid_details": []  # Armazena FID em épocas específicas
        }

        print(f"\n--- Starting experiment: {exp_name} ---")
        print(f"Config: {json.dumps({k: v for k, v in self.config.to_dict().items() if k != 'output_dir'}, indent=2)}")
        start_time = time.time()
        
        # Salva amostras reais para cálculo do FID
        self._save_real_samples()

        for epoch in range(self.config.num_epochs):
            epoch_start = time.time()
            d_loss, g_loss = trainer.train_epoch()
            epoch_time = time.time() - epoch_start

            epoch_details = {
                "epoch": epoch + 1,
                "d_loss": d_loss,
                "g_loss": g_loss,
                "time": epoch_time
            }
            results["epoch_details"].append(epoch_details)

            print(f"[Epoch {epoch+1}/{self.config.num_epochs}] D_loss: {d_loss:.4f} G_loss: {g_loss:.4f} Time: {epoch_time:.2f}s")
            
            # Calcula FID no intervalo definido ou na última época
            if (epoch + 1) % self.config.fid_eval_interval == 0 or (epoch + 1) == self.config.num_epochs:
                fid_dir = os.path.join(exp_dir, f"fid_samples_epoch_{epoch+1}")
                trainer.generate_samples(fid_dir, self.config.n_samples_save)
                fid_value = self._calculate_fid(fid_dir)
                
                # Remove amostras temporárias para economizar espaço
                shutil.rmtree(fid_dir)
                
                fid_details = {
                    "epoch": epoch + 1,
                    "fid": fid_value
                }
                results["fid_details"].append(fid_details)
                
                print(f"✅ FID @ epoch {epoch+1}: {fid_value:.2f}")
                
                # Atualiza melhor FID se necessário
                if fid_value < results["final_fid"]:
                    results["final_fid"] = fid_value
                    # Salva o modelo do gerador
                    torch.save(
                        trainer.generator.state_dict(),
                        os.path.join(exp_dir, "best_generator.pth")
                    )

        results["training_time"] = time.time() - start_time
        
        # Gera amostras finais para avaliação
        trainer.generate_samples(self.generated_dir, self.config.n_samples_save)
        final_fid = self._calculate_fid(self.generated_dir)
        results["final_fid"] = final_fid
        
        # Salva o modelo final
        torch.save(
            trainer.generator.state_dict(),
            os.path.join(exp_dir, "final_generator.pth")
        )
        torch.save(
            trainer.discriminator.state_dict(),
            os.path.join(exp_dir, "final_discriminator.pth")
        )
        
        # Gera gráficos
        self._plot_training_curves(results, exp_dir)
        
        # Salva resultados
        with open(os.path.join(exp_dir, "results.json"), "w") as f:
            json.dump(results, f, indent=2)

        print(f"✅ Experiment {exp_name} completed. Final FID: {final_fid:.2f}")
        return results

# MAIN

In [ ]:
class GridSearchRunner:
    def __init__(self, base_config):
        self.base_config = base_config
        self.results = []
        self.grid_params = {
            "latent_dim": [64],
            "lr_G": [0.0001],
            "lr_D": [0.0004],
            "disc_train_ratio": [1],
            "beta1": [(0.5, 0.999)]
        }
        
    def generate_configs(self):
        keys, values = zip(*self.grid_params.items())
        for combination in itertools.product(*values):
            yield dict(zip(keys, combination))
    
    def run(self):
        print(f"Starting grid search with {len(list(self.generate_configs()))} configurations")
        print(f"Search space: {json.dumps(self.grid_params, indent=2)}")
        
        best_fid = float('inf')
        best_config = None
        
        for i, params in enumerate(self.generate_configs()):
            # Create new config
            config = copy.deepcopy(self.base_config)
            config.update_from_dict(params)
            
            # Generate experiment name
            exp_name = f"exp_{i:03d}"
            for key, value in params.items():
                if key == "beta1":
                    value_str = f"{value[0]}_{value[1]}"
                else:
                    value_str = str(value)
                exp_name += f"_{key[:3]}_{value_str}"
            
            # Run experiment
            runner = ExperimentRunner(config)
            result = runner.run_experiment(config.seeds[0], exp_name)
            
            # Save result
            self.results.append({
                "params": params,
                "result": result,
                "exp_name": exp_name
            })
            
            # Update best result
            if result["final_fid"] < best_fid:
                best_fid = result["final_fid"]
                best_config = params
                print(f"🔥 New best FID: {best_fid:.2f} with config: {params}")
                    
        # Save grid search results
        summary = {
            "best_fid": best_fid,
            "best_config": best_config,
            "all_results": self.results
        }
        
        summary_path = os.path.join(self.base_config.output_dir, "grid_search_summary.json")
        with open(summary_path, "w") as f:
            json.dump(summary, f, indent=2)
            
        print(f"\n🎯 Grid search completed! Best FID: {best_fid:.2f}")
        print(f"Best configuration: {json.dumps(best_config, indent=2)}")
        print(f"Summary saved at: {summary_path}")
        
        return summary

if __name__ == "__main__":
    # Create output directory
    base_config = Config()
    base_config.output_dir = "./cgan_grid_search"
    os.makedirs(base_config.output_dir, exist_ok=True)
    
    # Run grid search
    grid_runner = GridSearchRunner(base_config)
    grid_runner.run()